# NonlinearDensity

A unified probabilistic primitive in GTSAM for representing a (possibly shifted) Gaussian density on a manifold value type. This notebook serves three audiences:

1. General GTSAM users who just want a probability-like object.
2. EKF users receiving `NonlinearDensity` outputs.
3. Advanced users leveraging transport, reset, and fusion of Left Extended Concentrated Gaussians (L-ECGs).

Related notebooks: {doc}`PriorFactor <PriorFactor.ipynb>`, {doc}`NonlinearLikelihood <NonlinearLikelihood.ipynb>`.

<a href="https://colab.research.google.com/github/borglab/gtsam/blob/develop/gtsam/nonlinear/doc/NonlinearDensity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install GTSAM and Plotly from pip if running in Google Colab
try:
    import google.colab
    %pip install --quiet gtsam-develop 
except ImportError:
    pass # Not in Colab

In [ ]:
import math
from typing import TypeAlias

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import gtsam
from gtsam import Point2, Point3, Pose2, Rot2

# Type aliases for clarity when passing to GTSAM
V: TypeAlias = np.ndarray  # Vector (1D) passed into GTSAM
M: TypeAlias = np.ndarray  # Matrix (2D) passed into GTSAM

## 1. General Usage

`NonlinearDensity<T>` behaves like a continuous probability density over a manifold variable `T`. For Gaussian noise models it evaluates to properly normalized probabilities (up to floating-point limits):

- Construct with origin `o`, covariance `Σ` (or noise model), and optional tangent mean `m`.
- Error term internally: `e(x) = -Local(x, o) - m` (if mean provided).
- Probability: `P(x) = exp( logProbability(x) )`.
- `evaluate(x)` is a convenience synonym for `P(x)`.

We'll start with a simple 2D example using `Point2` (Euclidean).

In [3]:
# Zero-mean Point2 density
origin: V = Point2(0.0, 0.0)
cov: M = np.array([[0.4, 0.1],
                   [0.1, 0.2]], dtype=float)
density = gtsam.NonlinearDensityPoint2(1, origin, cov)

xs: V = np.linspace(-2,2,160)
ys: V = np.linspace(-2,2,160)
X, Y = np.meshgrid(xs, ys)
Z: M = np.zeros_like(X)
for i in range(X.shape[0]):
    for j in range(X.shape[1]):
        Z[i,j] = density.evaluate(Point2(X[i,j], Y[i,j]))

# NOTE: In the Python wrapper Point2 is a numpy ndarray alias, so we use p[0], p[1].
fig = go.Figure()
fig.add_trace(go.Contour(x=xs, y=ys, z=Z, colorscale='Viridis', contours=dict(showlabels=True)))
fig.add_trace(go.Scatter(x=[origin[0]], y=[origin[1]], mode='markers', marker=dict(color='red', size=10), name='origin'))
fig.update_layout(title='Point2 NonlinearDensity (zero-mean)', xaxis_title='x', yaxis_title='y')
fig.show()

### Adding a Tangent-Space Mean

A non-zero mean shifts the mode to `origin.retract(mean)` (Point2: just addition).

In [4]:
# Non-zero tangent mean shifts the mode to origin.retract(mean)
mean: V = np.array([0.8, -0.4], dtype=float)
density_shifted = gtsam.NonlinearDensityPoint2(2, origin, cov, mean)
mode_point: V = origin + mean  # numpy-based Point2 + mean
mode_prob = density_shifted.evaluate(Point2(mode_point[0], mode_point[1]))
print('Mode probability (approx peak):', mode_prob)

Zs: M = np.zeros_like(Z)
for i in range(X.shape[0]):
    for j in range(X.shape[1]):
        Zs[i,j] = density_shifted.evaluate(Point2(X[i,j], Y[i,j]))

fig2 = go.Figure()
fig2.add_trace(go.Contour(x=xs, y=ys, z=Zs, colorscale='Inferno', contours=dict(showlabels=True)))
fig2.add_trace(go.Scatter(x=[origin[0]], y=[origin[1]], mode='markers', marker=dict(color='red', size=10), name='origin'))
fig2.add_trace(go.Scatter(x=[mode_point[0]], y=[mode_point[1]], mode='markers', marker=dict(color='green', size=10, symbol='x'), name='mode'))
fig2.update_layout(title='Shifted NonlinearDensity (Point2)', xaxis_title='x', yaxis_title='y')
fig2.show()

Mode probability (approx peak): 0.6015491419254174


### Pose2 Example with Samples (Banana Shape)

Let us simulate a vehicle taking N steps, but with large uncertainty in angle. To obtain a pronounced "banana" distribution we simulate many short forward motion increments with heading noise. Each sample is a random walk of small steps:

1. Start at identity pose.
2. For each of K steps, sample a small forward translation with little lateral drift and a heading perturbation.
3. Compose these incremental poses.
4. Record the final (x,y,theta).

Accumulated heading noise bends trajectories, producing a curved (banana-like) marginal over (x,y).

In [87]:
# Incremental random-walk generation of a banana-shaped distribution
rng = np.random.default_rng(42)
N: int = 5000   # number of trajectories
K: int = 20     # number of small increments per trajectory

# Increment noise parameters (tangent space): forward, lateral, heading
sigma_forward = 0.01   # small forward step ~ 10 cm
sigma_lateral = 0.01   # very little sideways motion
sigma_theta = np.deg2rad(5)  # modest heading noise each step (5 degrees)
forward_mean = 0.12    # deterministic forward progress per step

xs = []; ys = []; thetas = []
for _ in range(N):
    pose = Pose2(0,0,0)
    for _ in range(K):
        # Sample incremental tangent vector (dx_body, dy_body, dtheta)
        inc = np.array([
            rng.normal(forward_mean, sigma_forward),
            rng.normal(0.0,         sigma_lateral),
            rng.normal(0.0,         sigma_theta)
        ])
        pose = pose.retract(inc)
    xs.append(pose.x()); ys.append(pose.y()); thetas.append(pose.theta())

xs = np.asarray(xs); ys = np.asarray(ys); thetas = np.rad2deg(np.asarray(thetas))

fig3 = go.Figure()
fig3.add_trace(go.Scattergl(
    x=xs, y=ys, mode='markers',
    marker=dict(size=3, opacity=0.35, color=thetas, colorscale='Turbo', colorbar=dict(title='theta')),
    name='end poses'))
fig3.update_layout(
    title='Pose2 random-walk samples → pronounced banana distribution',
    xaxis_title='x', yaxis_title='y', yaxis_scaleanchor='x')
fig3.show()

Just as in the RSS 2008 paper, “The Banana Distribution is Gaussian”, we can now fit a Gaussian *to the exponential coordinates:

In [88]:
# Stack xs, ys, thetas into Pose2s, Logmap to tangent vectors, fit mean/covariance
tangent_vecs = np.array([Pose2.Logmap(Pose2(x, y, th)) for x, y, th in zip(xs, ys, np.deg2rad(thetas))])
mean_tangent = np.mean(tangent_vecs, axis=0)
cov_tangent = np.cov(tangent_vecs, rowvar=False)
print("Mean (tangent):", mean_tangent)
print("Covariance (tangent):\n", cov_tangent)

Mean (tangent): [2.36932908 0.00592982 0.00667413]
Covariance (tangent):
 [[ 0.002959    0.00033199 -0.00047491]
 [ 0.00033199  0.07399194 -0.00841532]
 [-0.00047491 -0.00841532  0.1487115 ]]


In [89]:
# Sample from the tangent Gaussian, push through ExpMap, and plot (x, y, theta)
samples_tangent = rng.multivariate_normal(mean_tangent, cov_tangent, size=N)

xs_push = []
ys_push = []
thetas_push = []
for v in samples_tangent:
  p = Pose2.Expmap(v)
  xs_push.append(p.x())
  ys_push.append(p.y())
  thetas_push.append(p.theta())

xs_push = np.asarray(xs_push)
ys_push = np.asarray(ys_push)
thetas_push = np.rad2deg(np.asarray(thetas_push))

fig_push2 = go.Figure()
fig_push2.add_trace(go.Scattergl(
  x=xs_push, y=ys_push, mode='markers',
  marker=dict(size=3, opacity=0.35, color=thetas_push, colorscale='Turbo', colorbar=dict(title='theta')),
  name='Expmap samples'))
fig_push2.update_layout(
  title='Expmap of Tangent Gaussian Samples (Pose2)',
  xaxis_title='x', yaxis_title='y', yaxis_scaleanchor='x')
fig_push2.show()

## 2. EKF Perspective

An Extended Kalman Filter (EKF) posterior can be represented as a `NonlinearDensity`:

- Origin: the linearization/reference state.
- Tangent mean: offset from origin to the mode.
- Covariance: uncertainty in that tangent space.

Before using this posterior as a prior for the next prediction, call `reset()` to move the origin to the mode and zero the mean (keeping covariance transported).

In [90]:
# Simulated EKF posterior with non-zero tangent mean
origin_post = Pose2(1.0, 2.0, 0.2)
cov_post: M = np.diag([0.05, 0.04, 0.02]).astype(float)
mean_post: V = np.array([0.3, -0.1, 0.15], dtype=float)
posterior = gtsam.NonlinearDensityPose2(5, origin_post, cov_post, mean_post)
mode_pose = posterior.retractMean()
print('Origin:', origin_post)
print('Mode  :', mode_pose)
print('Mode probability:', posterior.evaluate(mode_pose))

reset_density = posterior.reset()
print('After reset -> origin == mode?', reset_density.origin() == mode_pose)
print('Reset mean (should be empty/None):', reset_density.mean())

fig_ekf = go.Figure()
fig_ekf.add_trace(go.Scatter(x=[origin_post.x()], y=[origin_post.y()], mode='markers',
                             marker=dict(color='red', size=10), name='origin'))
fig_ekf.add_trace(go.Scatter(x=[mode_pose.x()], y=[mode_pose.y()], mode='markers',
                             marker=dict(color='green', size=10, symbol='x'), name='mode'))
fig_ekf.update_layout(title='EKF Posterior: Origin vs Mode', xaxis_title='x', yaxis_title='y')
fig_ekf.show()

Origin: (1, 2, 0.2)

Mode  : (1.31388691, 1.96159414, 0.35)

Mode probability: 9.769507109185751
After reset -> origin == mode? False
Reset mean (should be empty/None): None


## 3. Advanced Operations: L-ECG Mechanics

`NonlinearDensity` implements a Left Extended Concentrated Gaussian (L-ECG). Advanced capabilities:

1. `reset()` – Move origin to mode, zero the tangent mean, transport covariance.
2. `transportTo(x_hat)` – Express density in a new chart (new origin) inducing a mean and transported covariance.
3. `operator*` – Approximate fusion of two densities (same key) in a common chart followed by reset.

Below: transport and fusion examples (Pose2).

In [91]:
# Transport & Fusion with separated, differently oriented densities
originA = Pose2(-2.0, 0.0, math.pi/4)
covA: M = np.diag([0.25, 0.12, 0.10]).astype(float)
meanA: V = np.array([0.4, 0.0, 0.0], dtype=float)

originB = Pose2(2.0, 0.0, -math.pi/3)
covB: M = np.diag([0.25, 0.12, 0.10]).astype(float)
meanB: V = np.array([0.4, 0.0, 0.0], dtype=float)

dA = gtsam.NonlinearDensityPose2(10, originA, covA, meanA)
dB = gtsam.NonlinearDensityPose2(10, originB, covB, meanB)

fused = dA * dB
print('Fused origin:', fused.origin())
print('Fused mean (should be None):', fused.mean())

N_small: int = 1200
samplesA: M = np.random.multivariate_normal(meanA, covA, size=N_small)
samplesB: M = np.random.multivariate_normal(meanB, covB, size=N_small)
xyA = np.array([[originA.retract(v).x(), originA.retract(v).y(), originA.retract(v).theta()] for v in samplesA])
xyB = np.array([[originB.retract(v).x(), originB.retract(v).y(), originB.retract(v).theta()] for v in samplesB])

fig_fusion = go.Figure()
fig_fusion.add_trace(go.Scattergl(x=xyA[:,0], y=xyA[:,1], mode='markers',
                                  marker=dict(size=3, opacity=0.25, color=xyA[:,2], colorscale='Blues'), name='A samples'))
fig_fusion.add_trace(go.Scattergl(x=xyB[:,0], y=xyB[:,1], mode='markers',
                                  marker=dict(size=3, opacity=0.25, color=xyB[:,2], colorscale='Reds'), name='B samples'))

def heading_arrow(pose, length: float = 0.7, color: str = 'black', name: str = 'heading') -> None:
    x = pose.x(); y = pose.y(); th = pose.theta()
    x2 = x + length*math.cos(th)
    y2 = y + length*math.sin(th)
    fig_fusion.add_trace(go.Scatter(x=[x, x2], y=[y, y2], mode='lines', line=dict(color=color, width=3), name=name, showlegend=False))

fig_fusion.add_trace(go.Scatter(x=[originA.x()], y=[originA.y()], mode='markers', marker=dict(color='blue', size=10), name='Origin A'))
fig_fusion.add_trace(go.Scatter(x=[originB.x()], y=[originB.y()], mode='markers', marker=dict(color='red', size=10), name='Origin B'))
fig_fusion.add_trace(go.Scatter(x=[fused.origin().x()], y=[fused.origin().y()], mode='markers', marker=dict(color='green', size=12, symbol='x'), name='Fused'))

heading_arrow(originA, color='blue')
heading_arrow(originB, color='red')
heading_arrow(fused.origin(), color='green')

fig_fusion.update_layout(title='Fusion of Two Opposed Pose2 Densities (Result ~ centered, heading ~ 0)',
                         xaxis_title='x', yaxis_title='y', yaxis_scaleanchor='x')
fig_fusion.show()

Fused origin: (0.320684765, 0.704438054, -0.130899694)

Fused mean (should be None): None


### Caveats

- Fusion and transport are first-order; for large separations, consider iterative refinement.
- Large tangent means may signal re-centering (`reset`) to maintain Gaussian fidelity.
- Current examples track covariance externally for sampling; exposing underlying Gaussian model in Python would make advanced visualization easier.

## Summary

`NonlinearDensity` = manifold-aware Gaussian density: basic probability evaluation (Section 1), EKF posterior handling with `retractMean()`/`reset()` (Section 2), and geometric operations (transport, fusion) for advanced workflows (Section 3). Choose the simplest interface that solves your task.